# 7. Squat TCN Training (Colab)

This notebook trains a squat-only `TCN / 1D CNN` rep-count regressor on top of the engineered squat features produced by Colab 5.

Pipeline:

`video -> YOLO pose -> squat features -> TCN -> predicted rep count`

This notebook is designed to keep artifacts on Google Drive so the trained outputs persist across sessions.

## 0. Connect Google Drive

Mount Drive first so the project data and training outputs are written to persistent storage instead of temporary `/content`.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Paths and Script Sync

Set the code root and the Drive-backed project root. The trainer script is copied into the Drive project tree so the run is fully self-contained.

In [2]:
from pathlib import Path
import shutil

CODE_ROOT = Path('/content/CV_Image_pose_detection')
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection')

TRAINER_REL = Path('artifacts/3_Modeling/train_squat_tcn.py')
TRAINER_SRC = CODE_ROOT / TRAINER_REL
TRAINER_DST = DRIVE_PROJECT_ROOT / TRAINER_REL

if TRAINER_SRC.exists():
    TRAINER_DST.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(TRAINER_SRC, TRAINER_DST)

ANNOTATION_DIR = DRIVE_PROJECT_ROOT / 'Data/LLSP/annotation_cleaned'
FEATURE_INDEX = ANNOTATION_DIR / 'squat_feature_index.csv'
FEATURE_DIR = ANNOTATION_DIR / 'squat_features'

print('CODE_ROOT =', CODE_ROOT)
print('DRIVE_PROJECT_ROOT =', DRIVE_PROJECT_ROOT)
print('TRAINER_DST =', TRAINER_DST)
print('FEATURE_INDEX exists =', FEATURE_INDEX.exists())
print('FEATURE_DIR exists =', FEATURE_DIR.exists())

CODE_ROOT = /content/CV_Image_pose_detection
DRIVE_PROJECT_ROOT = /content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection
TRAINER_DST = /content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/artifacts/3_Modeling/train_squat_tcn.py
FEATURE_INDEX exists = True
FEATURE_DIR exists = True


## 2. Runtime Check

GPU is preferred for faster training, but CPU still works.

In [3]:
import torch
print('cuda_available =', torch.cuda.is_available())
print('device_count =', torch.cuda.device_count())
print('device_name =', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

cuda_available = False
device_count = 0
device_name = cpu


## 3. Optional Dependency Check

Colab usually includes `numpy` and `torch`. Run this only if the training cell reports a missing package.

In [4]:
# !pip install numpy torch

## 4. Train the Squat TCN

This launches the standalone trainer against the Drive-backed project root, so outputs are written under:

`artifacts/3_Modeling/training_outputs/<run_name>/`

In [5]:
RUN_NAME = 'squat_tcn_v1'
SEQ_LEN = 128
EPOCHS = 80
BATCH_SIZE = 16
LR = 1e-3
WEIGHT_DECAY = 1e-4
CHANNELS = 64
KERNEL_SIZE = 3
NUM_BLOCKS = 4
DROPOUT = 0.2
PATIENCE = 15
LOSS = 'smooth_l1'
EVAL_TRANSFORM = 'raw'
SELECTION_METRIC = 'mae'

In [6]:
!python {TRAINER_DST} \
  --project-dir {DRIVE_PROJECT_ROOT} \
  --run-name {RUN_NAME} \
  --seq-len {SEQ_LEN} \
  --epochs {EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --lr {LR} \
  --weight-decay {WEIGHT_DECAY} \
  --channels {CHANNELS} \
  --kernel-size {KERNEL_SIZE} \
  --num-blocks {NUM_BLOCKS} \
  --dropout {DROPOUT} \
  --patience {PATIENCE} \
  --loss {LOSS} \
  --eval-transform {EVAL_TRANSFORM} \
  --selection-metric {SELECTION_METRIC} \
  --device cuda

[001] train_mae=11.3080 valid_mae=12.7955 valid_within_1=0.0625 selection(mae)=12.7955
[002] train_mae=9.1086 valid_mae=10.1669 valid_within_1=0.0625 selection(mae)=10.1669
[003] train_mae=7.9017 valid_mae=10.1066 valid_within_1=0.0625 selection(mae)=10.1066
[004] train_mae=8.0794 valid_mae=10.2505 valid_within_1=0.1250 selection(mae)=10.2505
[005] train_mae=7.2600 valid_mae=7.1635 valid_within_1=0.0625 selection(mae)=7.1635
[006] train_mae=6.3515 valid_mae=6.4480 valid_within_1=0.2500 selection(mae)=6.4480
[007] train_mae=4.6668 valid_mae=5.6994 valid_within_1=0.2500 selection(mae)=5.6994
[008] train_mae=3.7573 valid_mae=5.5768 valid_within_1=0.1875 selection(mae)=5.5768
[009] train_mae=4.0585 valid_mae=5.3535 valid_within_1=0.1250 selection(mae)=5.3535
[010] train_mae=3.5751 valid_mae=6.0347 valid_within_1=0.1875 selection(mae)=6.0347
[011] train_mae=3.7842 valid_mae=4.4877 valid_within_1=0.1250 selection(mae)=4.4877
[012] train_mae=3.4474 valid_mae=5.8958 valid_within_1=0.1875 selec

## 5. Inspect Saved Outputs

After training, inspect the saved metrics and predictions.

In [7]:
OUTPUT_DIR = DRIVE_PROJECT_ROOT / 'artifacts/3_Modeling/training_outputs' / RUN_NAME
print('OUTPUT_DIR =', OUTPUT_DIR)
list(OUTPUT_DIR.iterdir()) if OUTPUT_DIR.exists() else 'missing output dir'

OUTPUT_DIR = /content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/artifacts/3_Modeling/training_outputs/squat_tcn_v1


[PosixPath('/content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/artifacts/3_Modeling/training_outputs/squat_tcn_v1/history.csv'),
 PosixPath('/content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/artifacts/3_Modeling/training_outputs/squat_tcn_v1/metrics_summary.json'),
 PosixPath('/content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/artifacts/3_Modeling/training_outputs/squat_tcn_v1/feature_std.npy'),
 PosixPath('/content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/artifacts/3_Modeling/training_outputs/squat_tcn_v1/config.json'),
 PosixPath('/content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/artifacts/3_Modeling/training_outputs/squat_tcn_v1/feature_mean.npy'),
 PosixPath('/content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/artifacts/3_Modeling/training_outputs/squat_tcn_v1/predictions.csv'),
 PosixPath('/content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/artifacts/3_Modeling/training_outputs/squat_tcn_v1/best_model.pt

In [8]:
import json
from pprint import pprint

metrics_path = OUTPUT_DIR / 'metrics_summary.json'
if metrics_path.exists():
    with open(metrics_path, 'r', encoding='utf-8') as f:
        pprint(json.load(f))
else:
    print('metrics_summary.json not found')

{'best_epoch': 60,
 'best_selection_metric': 2.3097587525844574,
 'train_metrics': {'mae': 1.6943411990707995,
                   'rmse': 2.6325308104345075,
                   'rows': 102.0,
                   'within_1': 0.47058823529411764},
 'valid_metrics': {'mae': 2.3097587525844574,
                   'rmse': 3.0691849920429424,
                   'rows': 16.0,
                   'within_1': 0.3125}}


## 6. Interpreting the Metrics

- `MAE` reflects the **average counting error** across samples.
- `Within-1` reflects how often the prediction is **near-exact**, meaning within one repetition of the true count.

These two metrics can move in different directions. `MAE` can improve while `Within-1` decreases if the model reduces large errors overall but produces fewer predictions inside the strict `+/-1` repetition band.

When to use `Average`:
- When variability is high, but the average converges toward the truth.
- This is more useful when reliability over many samples matters more than exact precision on each individual sample.

When to use `Near-Exact`:
- When high confidence is needed on a single sample, and the extra cost or complexity is justified.
- This is more useful when exact or almost exact counts matter more than average error reduction.

For this notebook, the next step after reading the metrics is to inspect the `valid` rows in `predictions.csv` and determine whether the lower `Within-1` score comes from moderate misses that are still close to the true count, or from a few larger failures.

If the metric you care about is `Within-1`, use:
- `EVAL_TRANSFORM = 'round_clip_nonneg'`
- `SELECTION_METRIC = 'within_1'`

If the metric you care about is average error, keep:
- `EVAL_TRANSFORM = 'raw'`
- `SELECTION_METRIC = 'mae'`

In [9]:
import pandas as pd

pred_path = OUTPUT_DIR / 'predictions.csv'
if pred_path.exists():
    pred_df = pd.read_csv(pred_path)
    display(pred_df.head())
    display(pred_df.groupby('split')['abs_error'].agg(['count', 'mean']))
else:
    print('predictions.csv not found')

,name,split,true_count,raw_pred_count,eval_pred_count,abs_error
0,stu6_58.mp4,train,5.0,4.117399,4.117399,0.882601
1,stu6_60.mp4,train,35.0,35.680401,35.680401,0.680401
2,stu4_67.mp4,train,9.0,10.492644,10.492644,1.492644
3,stu5_65.mp4,train,18.0,18.117275,18.117275,0.117275
4,test2295.mp4,train,1.0,1.509711,1.509711,0.509711


,count,mean
split,,
train,102,1.694341
valid,16,2.309759


In [10]:
if pred_path.exists():
    valid_df = pred_df.loc[pred_df['split'] == 'valid'].copy()
    display(valid_df.sort_values('abs_error', ascending=False).head(10))
else:
    print('predictions.csv not found')

,name,split,true_count,raw_pred_count,eval_pred_count,abs_error
105,stu7_66.mp4,valid,13.0,4.556545,4.556545,8.443455
104,stu6_57.mp4,valid,15.0,19.712650,19.712650,4.712650
117,stu10_69.mp4,valid,39.0,42.857937,42.857937,3.857937
115,test2349.mp4,valid,6.0,2.630744,2.630744,3.369256
111,stu5_71.mp4,valid,19.0,16.498466,16.498466,2.501534
116,stu9_67.mp4,valid,0.0,2.487211,2.487211,2.487211
106,stu6_61.mp4,valid,19.0,21.254414,21.254414,2.254414
112,stu10_67.mp4,valid,6.0,3.791375,3.791375,2.208625
108,train3921.mp4,valid,4.0,2.284456,2.284456,1.715544
113,stu7_65.mp4,valid,9.0,7.355286,7.355286,1.644714


### Decision Note: Right Model vs Current Performance

At this stage there are two different questions:

1. **Is `TCN / 1D CNN` the right model family for this pipeline?**
   - For the current `YOLO pose -> feature sequence -> count prediction` setup, the answer is **probably yes**.
   - It is well aligned with temporal pose features and is a reasonable learned replacement for the FSM backend.

2. **Is this specific trained TCN performing well enough yet?**
   - Not necessarily.
   - The current run is promising, but it is not yet enough to conclude that the model has reached its best performance.

The practical interpretation is:
- do not discard the TCN architecture after one run
- inspect the `valid` errors in `predictions.csv`
- run a small number of controlled follow-ups before deciding whether the issue is model choice or model tuning

In other words, the current question is less **"Is TCN wrong?"** and more **"Has this TCN been tuned enough to show its real potential?"**

## 7. Controlled TCN Follow-Ups

Run one or two focused follow-up experiments instead of making many changes at once.

Suggested controlled follow-ups:
- change `seq_len` while keeping the rest fixed
- change the regression loss while keeping the architecture fixed

The controlled follow-ups below are:
- `seq_len = 192` with the same baseline settings
- `loss = l1` with the same baseline settings
- `Within-1`-focused checkpoint selection with rounded nonnegative evaluation counts

In [11]:
FOLLOW_UP_RUNS = [
    {
        'run_name': 'squat_tcn_seq192',
        'seq_len': 192,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'lr': LR,
        'weight_decay': WEIGHT_DECAY,
        'channels': CHANNELS,
        'kernel_size': KERNEL_SIZE,
        'num_blocks': NUM_BLOCKS,
        'dropout': DROPOUT,
        'patience': PATIENCE,
        'loss': 'smooth_l1',
        'eval_transform': 'raw',
        'selection_metric': 'mae',
    },
    {
        'run_name': 'squat_tcn_l1loss',
        'seq_len': SEQ_LEN,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'lr': LR,
        'weight_decay': WEIGHT_DECAY,
        'channels': CHANNELS,
        'kernel_size': KERNEL_SIZE,
        'num_blocks': NUM_BLOCKS,
        'dropout': DROPOUT,
        'patience': PATIENCE,
        'loss': 'l1',
        'eval_transform': 'raw',
        'selection_metric': 'mae',
    },
    {
        'run_name': 'squat_tcn_within1',
        'seq_len': SEQ_LEN,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'lr': LR,
        'weight_decay': WEIGHT_DECAY,
        'channels': CHANNELS,
        'kernel_size': KERNEL_SIZE,
        'num_blocks': NUM_BLOCKS,
        'dropout': DROPOUT,
        'patience': PATIENCE,
        'loss': 'smooth_l1',
        'eval_transform': 'round_clip_nonneg',
        'selection_metric': 'within_1',
    },
]

FOLLOW_UP_RUNS

[{'run_name': 'squat_tcn_seq192',
  'seq_len': 192,
  'epochs': 80,
  'batch_size': 16,
  'lr': 0.001,
  'weight_decay': 0.0001,
  'channels': 64,
  'kernel_size': 3,
  'num_blocks': 4,
  'dropout': 0.2,
  'patience': 15,
  'loss': 'smooth_l1',
  'eval_transform': 'raw',
  'selection_metric': 'mae'},
 {'run_name': 'squat_tcn_l1loss',
  'seq_len': 128,
  'epochs': 80,
  'batch_size': 16,
  'lr': 0.001,
  'weight_decay': 0.0001,
  'channels': 64,
  'kernel_size': 3,
  'num_blocks': 4,
  'dropout': 0.2,
  'patience': 15,
  'loss': 'l1',
  'eval_transform': 'raw',
  'selection_metric': 'mae'},
 {'run_name': 'squat_tcn_within1',
  'seq_len': 128,
  'epochs': 80,
  'batch_size': 16,
  'lr': 0.001,
  'weight_decay': 0.0001,
  'channels': 64,
  'kernel_size': 3,
  'num_blocks': 4,
  'dropout': 0.2,
  'patience': 15,
  'loss': 'smooth_l1',
  'eval_transform': 'round_clip_nonneg',
  'selection_metric': 'within_1'}]

In [12]:
import subprocess

for cfg in FOLLOW_UP_RUNS:
    cmd = [
        'python', str(TRAINER_DST),
        '--project-dir', str(DRIVE_PROJECT_ROOT),
        '--run-name', cfg['run_name'],
        '--seq-len', str(cfg['seq_len']),
        '--epochs', str(cfg['epochs']),
        '--batch-size', str(cfg['batch_size']),
        '--lr', str(cfg['lr']),
        '--weight-decay', str(cfg['weight_decay']),
        '--channels', str(cfg['channels']),
        '--kernel-size', str(cfg['kernel_size']),
        '--num-blocks', str(cfg['num_blocks']),
        '--dropout', str(cfg['dropout']),
        '--patience', str(cfg['patience']),
        '--loss', cfg['loss'],
        '--eval-transform', cfg['eval_transform'],
        '--selection-metric', cfg['selection_metric'],
        '--device', 'cuda',
    ]
    print('\nRunning:', ' '.join(cmd))
    subprocess.run(cmd, check=True)



Running: python /content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/artifacts/3_Modeling/train_squat_tcn.py --project-dir /content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection --run-name squat_tcn_seq192 --seq-len 192 --epochs 80 --batch-size 16 --lr 0.001 --weight-decay 0.0001 --channels 64 --kernel-size 3 --num-blocks 4 --dropout 0.2 --patience 15 --loss smooth_l1 --eval-transform raw --selection-metric mae --device cuda

Running: python /content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/artifacts/3_Modeling/train_squat_tcn.py --project-dir /content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection --run-name squat_tcn_l1loss --seq-len 128 --epochs 80 --batch-size 16 --lr 0.001 --weight-decay 0.0001 --channels 64 --kernel-size 3 --num-blocks 4 --dropout 0.2 --patience 15 --loss l1 --eval-transform raw --selection-metric mae --device cuda

Running: python /content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/artifacts/3_Modeling/train_squat_tcn

In [13]:
import json
import pandas as pd

comparison_rows = []

all_runs = [{
    'run_name': RUN_NAME,
    'seq_len': SEQ_LEN,
    'loss': LOSS,
    'eval_transform': EVAL_TRANSFORM,
    'selection_metric': SELECTION_METRIC,
}] + FOLLOW_UP_RUNS
for cfg in all_runs:
    metrics_path = DRIVE_PROJECT_ROOT / 'artifacts/3_Modeling/training_outputs' / cfg['run_name'] / 'metrics_summary.json'
    if not metrics_path.exists():
        continue
    with open(metrics_path, 'r', encoding='utf-8') as f:
        metrics = json.load(f)
    comparison_rows.append({
        'run_name': cfg['run_name'],
        'seq_len': cfg.get('seq_len'),
        'loss': cfg.get('loss'),
        'eval_transform': cfg.get('eval_transform'),
        'selection_metric': cfg.get('selection_metric'),
        'best_epoch': metrics.get('best_epoch'),
        'train_mae': metrics['train_metrics']['mae'],
        'train_rmse': metrics['train_metrics']['rmse'],
        'train_within_1': metrics['train_metrics']['within_1'],
        'valid_mae': metrics['valid_metrics']['mae'],
        'valid_rmse': metrics['valid_metrics']['rmse'],
        'valid_within_1': metrics['valid_metrics']['within_1'],
    })

comparison_df = pd.DataFrame(comparison_rows)
if not comparison_df.empty:
    display(comparison_df.sort_values(['valid_mae', 'valid_rmse', 'valid_within_1'], ascending=[True, True, False]))
else:
    print('No metrics_summary.json files found yet for the requested runs.')

,run_name,seq_len,loss,eval_transform,selection_metric,best_epoch,train_mae,train_rmse,train_within_1,valid_mae,valid_rmse,valid_within_1
0,squat_tcn_v1,128,smooth_l1,raw,mae,60,1.694341,2.632531,0.470588,2.309759,3.069185,0.3125
2,squat_tcn_l1loss,128,l1,raw,mae,51,1.619774,2.960265,0.627451,2.380133,3.049104,0.2500
1,squat_tcn_seq192,192,smooth_l1,raw,mae,48,1.568747,2.752886,0.558824,2.409282,3.197828,0.4375
3,squat_tcn_within1,128,smooth_l1,round_clip_nonneg,within_1,7,3.980392,5.819170,0.245098,5.750000,8.667468,0.3750


### 8. Reading the Follow-Up Results

Use the comparison table to answer three questions:

1. **Which TCN variant is currently best overall?**
   - If `squat_tcn_l1loss` has the best validation `MAE` and `RMSE`, then the `L1` loss is currently the strongest learned baseline.

2. **Did sequence length help?**
   - If `squat_tcn_seq192` does not improve validation metrics, then increasing `seq_len` is not currently justified.

3. **Did `Within-1`-focused checkpoint selection help?**
   - If `squat_tcn_within1` performs worse, then selecting checkpoints by `Within-1` with rounded nonnegative evaluation counts is too unstable for this setup.

If your current runs match that pattern, the practical conclusion is:
- the `TCN / 1D CNN` model family is still a valid direction
- the best learned squat baseline so far is likely `TCN + L1 loss`
- the `Within-1`-focused selection strategy should not be prioritized further right now
- future tuning should continue from the strongest run instead of from the weaker variants

## 9. Recommended Next-Step TCN Runs

These runs continue from the current strongest learned baseline (`TCN + L1 loss`) instead of revisiting weaker directions.

Recommended next steps:
- increase `patience` to check whether the best `L1` run is still improving late
- slightly widen the model to test whether the current TCN is underpowered

These are intentionally small, controlled changes.

In [14]:
RECOMMENDED_RUNS = [
    {
        'run_name': 'squat_tcn_l1_patience30',
        'seq_len': SEQ_LEN,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'lr': LR,
        'weight_decay': WEIGHT_DECAY,
        'channels': CHANNELS,
        'kernel_size': KERNEL_SIZE,
        'num_blocks': NUM_BLOCKS,
        'dropout': DROPOUT,
        'patience': 30,
        'loss': 'l1',
        'eval_transform': 'raw',
        'selection_metric': 'mae',
    },
    {
        'run_name': 'squat_tcn_l1_channels96',
        'seq_len': SEQ_LEN,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'lr': LR,
        'weight_decay': WEIGHT_DECAY,
        'channels': 96,
        'kernel_size': KERNEL_SIZE,
        'num_blocks': NUM_BLOCKS,
        'dropout': DROPOUT,
        'patience': PATIENCE,
        'loss': 'l1',
        'eval_transform': 'raw',
        'selection_metric': 'mae',
    },
]

RECOMMENDED_RUNS

[{'run_name': 'squat_tcn_l1_patience30',
  'seq_len': 128,
  'epochs': 80,
  'batch_size': 16,
  'lr': 0.001,
  'weight_decay': 0.0001,
  'channels': 64,
  'kernel_size': 3,
  'num_blocks': 4,
  'dropout': 0.2,
  'patience': 30,
  'loss': 'l1',
  'eval_transform': 'raw',
  'selection_metric': 'mae'},
 {'run_name': 'squat_tcn_l1_channels96',
  'seq_len': 128,
  'epochs': 80,
  'batch_size': 16,
  'lr': 0.001,
  'weight_decay': 0.0001,
  'channels': 96,
  'kernel_size': 3,
  'num_blocks': 4,
  'dropout': 0.2,
  'patience': 15,
  'loss': 'l1',
  'eval_transform': 'raw',
  'selection_metric': 'mae'}]

In [15]:
for cfg in RECOMMENDED_RUNS:
    cmd = [
        'python', str(TRAINER_DST),
        '--project-dir', str(DRIVE_PROJECT_ROOT),
        '--run-name', cfg['run_name'],
        '--seq-len', str(cfg['seq_len']),
        '--epochs', str(cfg['epochs']),
        '--batch-size', str(cfg['batch_size']),
        '--lr', str(cfg['lr']),
        '--weight-decay', str(cfg['weight_decay']),
        '--channels', str(cfg['channels']),
        '--kernel-size', str(cfg['kernel_size']),
        '--num-blocks', str(cfg['num_blocks']),
        '--dropout', str(cfg['dropout']),
        '--patience', str(cfg['patience']),
        '--loss', cfg['loss'],
        '--eval-transform', cfg['eval_transform'],
        '--selection-metric', cfg['selection_metric'],
        '--device', 'cuda',
    ]
    print('\nRunning:', ' '.join(cmd))
    subprocess.run(cmd, check=True)



Running: python /content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/artifacts/3_Modeling/train_squat_tcn.py --project-dir /content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection --run-name squat_tcn_l1_patience30 --seq-len 128 --epochs 80 --batch-size 16 --lr 0.001 --weight-decay 0.0001 --channels 64 --kernel-size 3 --num-blocks 4 --dropout 0.2 --patience 30 --loss l1 --eval-transform raw --selection-metric mae --device cuda

Running: python /content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/artifacts/3_Modeling/train_squat_tcn.py --project-dir /content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection --run-name squat_tcn_l1_channels96 --seq-len 128 --epochs 80 --batch-size 16 --lr 0.001 --weight-decay 0.0001 --channels 96 --kernel-size 3 --num-blocks 4 --dropout 0.2 --patience 15 --loss l1 --eval-transform raw --selection-metric mae --device cuda


In [16]:
recommended_rows = []
baseline_and_recommended = [
    {
        'run_name': 'squat_tcn_l1loss',
        'seq_len': SEQ_LEN,
        'loss': 'l1',
        'channels': CHANNELS,
        'patience': PATIENCE,
    }
] + RECOMMENDED_RUNS

for cfg in baseline_and_recommended:
    metrics_path = DRIVE_PROJECT_ROOT / 'artifacts/3_Modeling/training_outputs' / cfg['run_name'] / 'metrics_summary.json'
    if not metrics_path.exists():
        continue
    with open(metrics_path, 'r', encoding='utf-8') as f:
        metrics = json.load(f)
    recommended_rows.append({
        'run_name': cfg['run_name'],
        'channels': cfg.get('channels'),
        'patience': cfg.get('patience'),
        'loss': cfg.get('loss'),
        'best_epoch': metrics.get('best_epoch'),
        'valid_mae': metrics['valid_metrics']['mae'],
        'valid_rmse': metrics['valid_metrics']['rmse'],
        'valid_within_1': metrics['valid_metrics']['within_1'],
    })

recommended_df = pd.DataFrame(recommended_rows)
if not recommended_df.empty:
    display(recommended_df.sort_values(['valid_mae', 'valid_rmse', 'valid_within_1'], ascending=[True, True, False]))
else:
    print('No metrics found yet for the recommended next-step runs.')

,run_name,channels,patience,loss,best_epoch,valid_mae,valid_rmse,valid_within_1
2,squat_tcn_l1_channels96,96,15,l1,50,2.178970,2.903253,0.3125
0,squat_tcn_l1loss,64,15,l1,51,2.380133,3.049104,0.2500
1,squat_tcn_l1_patience30,64,30,l1,51,2.380133,3.049104,0.2500


## 10. Additional Targeted Tuning Around the Best Run

If `squat_tcn_l1_channels96` is currently the best-balanced run, the next sensible step is to tune around that configuration instead of returning to weaker settings.

These extra runs stay narrow and controlled:
- keep `L1` loss
- keep the stronger `96` channels
- test learning rate and dropout around the current best setup

In [17]:
TARGETED_RUNS = [
    {
        'run_name': 'squat_tcn_l1_channels96_lr5e4',
        'seq_len': SEQ_LEN,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'lr': 5e-4,
        'weight_decay': WEIGHT_DECAY,
        'channels': 96,
        'kernel_size': KERNEL_SIZE,
        'num_blocks': NUM_BLOCKS,
        'dropout': DROPOUT,
        'patience': PATIENCE,
        'loss': 'l1',
        'eval_transform': 'raw',
        'selection_metric': 'mae',
    },
    {
        'run_name': 'squat_tcn_l1_channels96_dropout01',
        'seq_len': SEQ_LEN,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'lr': LR,
        'weight_decay': WEIGHT_DECAY,
        'channels': 96,
        'kernel_size': KERNEL_SIZE,
        'num_blocks': NUM_BLOCKS,
        'dropout': 0.1,
        'patience': PATIENCE,
        'loss': 'l1',
        'eval_transform': 'raw',
        'selection_metric': 'mae',
    },
    {
        'run_name': 'squat_tcn_l1_channels128',
        'seq_len': SEQ_LEN,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'lr': LR,
        'weight_decay': WEIGHT_DECAY,
        'channels': 128,
        'kernel_size': KERNEL_SIZE,
        'num_blocks': NUM_BLOCKS,
        'dropout': DROPOUT,
        'patience': PATIENCE,
        'loss': 'l1',
        'eval_transform': 'raw',
        'selection_metric': 'mae',
    },
]

TARGETED_RUNS

[{'run_name': 'squat_tcn_l1_channels96_lr5e4',
  'seq_len': 128,
  'epochs': 80,
  'batch_size': 16,
  'lr': 0.0005,
  'weight_decay': 0.0001,
  'channels': 96,
  'kernel_size': 3,
  'num_blocks': 4,
  'dropout': 0.2,
  'patience': 15,
  'loss': 'l1',
  'eval_transform': 'raw',
  'selection_metric': 'mae'},
 {'run_name': 'squat_tcn_l1_channels96_dropout01',
  'seq_len': 128,
  'epochs': 80,
  'batch_size': 16,
  'lr': 0.001,
  'weight_decay': 0.0001,
  'channels': 96,
  'kernel_size': 3,
  'num_blocks': 4,
  'dropout': 0.1,
  'patience': 15,
  'loss': 'l1',
  'eval_transform': 'raw',
  'selection_metric': 'mae'},
 {'run_name': 'squat_tcn_l1_channels128',
  'seq_len': 128,
  'epochs': 80,
  'batch_size': 16,
  'lr': 0.001,
  'weight_decay': 0.0001,
  'channels': 128,
  'kernel_size': 3,
  'num_blocks': 4,
  'dropout': 0.2,
  'patience': 15,
  'loss': 'l1',
  'eval_transform': 'raw',
  'selection_metric': 'mae'}]

In [18]:
for cfg in TARGETED_RUNS:
    cmd = [
        'python', str(TRAINER_DST),
        '--project-dir', str(DRIVE_PROJECT_ROOT),
        '--run-name', cfg['run_name'],
        '--seq-len', str(cfg['seq_len']),
        '--epochs', str(cfg['epochs']),
        '--batch-size', str(cfg['batch_size']),
        '--lr', str(cfg['lr']),
        '--weight-decay', str(cfg['weight_decay']),
        '--channels', str(cfg['channels']),
        '--kernel-size', str(cfg['kernel_size']),
        '--num-blocks', str(cfg['num_blocks']),
        '--dropout', str(cfg['dropout']),
        '--patience', str(cfg['patience']),
        '--loss', cfg['loss'],
        '--eval-transform', cfg['eval_transform'],
        '--selection-metric', cfg['selection_metric'],
        '--device', 'cuda',
    ]
    print('\nRunning:', ' '.join(cmd))
    subprocess.run(cmd, check=True)



Running: python /content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/artifacts/3_Modeling/train_squat_tcn.py --project-dir /content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection --run-name squat_tcn_l1_channels96_lr5e4 --seq-len 128 --epochs 80 --batch-size 16 --lr 0.0005 --weight-decay 0.0001 --channels 96 --kernel-size 3 --num-blocks 4 --dropout 0.2 --patience 15 --loss l1 --eval-transform raw --selection-metric mae --device cuda

Running: python /content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/artifacts/3_Modeling/train_squat_tcn.py --project-dir /content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection --run-name squat_tcn_l1_channels96_dropout01 --seq-len 128 --epochs 80 --batch-size 16 --lr 0.001 --weight-decay 0.0001 --channels 96 --kernel-size 3 --num-blocks 4 --dropout 0.1 --patience 15 --loss l1 --eval-transform raw --selection-metric mae --device cuda

Running: python /content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/artifacts/3_

In [19]:
targeted_rows = []
best_family_runs = [
    {
        'run_name': 'squat_tcn_l1_channels96',
        'channels': 96,
        'lr': LR,
        'dropout': DROPOUT,
        'loss': 'l1',
    }
] + TARGETED_RUNS

for cfg in best_family_runs:
    metrics_path = DRIVE_PROJECT_ROOT / 'artifacts/3_Modeling/training_outputs' / cfg['run_name'] / 'metrics_summary.json'
    if not metrics_path.exists():
        continue
    with open(metrics_path, 'r', encoding='utf-8') as f:
        metrics = json.load(f)
    targeted_rows.append({
        'run_name': cfg['run_name'],
        'channels': cfg.get('channels'),
        'lr': cfg.get('lr'),
        'dropout': cfg.get('dropout'),
        'loss': cfg.get('loss'),
        'best_epoch': metrics.get('best_epoch'),
        'valid_mae': metrics['valid_metrics']['mae'],
        'valid_rmse': metrics['valid_metrics']['rmse'],
        'valid_within_1': metrics['valid_metrics']['within_1'],
    })

targeted_df = pd.DataFrame(targeted_rows)
if not targeted_df.empty:
    display(targeted_df.sort_values(['valid_mae', 'valid_rmse', 'valid_within_1'], ascending=[True, True, False]))
else:
    print('No metrics found yet for the targeted tuning runs.')

,run_name,channels,lr,dropout,loss,best_epoch,valid_mae,valid_rmse,valid_within_1
2,squat_tcn_l1_channels96_dropout01,96,0.0010,0.1,l1,48,2.132887,3.099379,0.5000
0,squat_tcn_l1_channels96,96,0.0010,0.2,l1,50,2.178970,2.903253,0.3125
3,squat_tcn_l1_channels128,128,0.0010,0.2,l1,59,2.529083,2.966686,0.1875
1,squat_tcn_l1_channels96_lr5e4,96,0.0005,0.2,l1,66,2.559521,3.536336,0.2500


## 11. Failure Analysis for the Best Run

The next step is to inspect the strongest learned squat model in more detail instead of continuing to tune blindly.

The current default best run for this section is `squat_tcn_l1_channels96_dropout01`, which produced the best balanced validation result in the latest targeted tuning: lower `MAE` than the earlier `channels96` run, similar `RMSE`, and better `Within-1` than the lower-learning-rate and larger-width alternatives.

Interpretation goal:
- if the worst `valid` errors line up with low `frames_valid`, low `mean_conf`, or low `valid_ratio`, then remaining mistakes are still tied to pose / feature quality.
- if the worst `valid` errors happen even on videos with strong support and confidence, then the remaining gap is more likely due to model behavior or target ambiguity than weak pose input.

This section focuses on the current best learned run and asks:
- which `valid` videos are hardest?
- are the worst errors associated with weak pose support or low confidence?
- are the remaining mistakes mostly moderate misses or a few severe failures?

In [20]:
BEST_TCN_RUN = 'squat_tcn_l1_channels96_dropout01'
BEST_OUTPUT_DIR = DRIVE_PROJECT_ROOT / 'artifacts/3_Modeling/training_outputs' / BEST_TCN_RUN
BEST_PRED_PATH = BEST_OUTPUT_DIR / 'predictions.csv'
FEATURE_SUMMARY_PATH = DRIVE_PROJECT_ROOT / 'Data/LLSP/annotation_cleaned/squat_feature_summary.csv'

print('BEST_OUTPUT_DIR =', BEST_OUTPUT_DIR)
print('BEST_PRED_PATH exists =', BEST_PRED_PATH.exists())
print('FEATURE_SUMMARY_PATH exists =', FEATURE_SUMMARY_PATH.exists())

BEST_OUTPUT_DIR = /content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/artifacts/3_Modeling/training_outputs/squat_tcn_l1_channels96
BEST_PRED_PATH exists = True
FEATURE_SUMMARY_PATH exists = True


In [21]:
if BEST_PRED_PATH.exists() and FEATURE_SUMMARY_PATH.exists():
    best_pred_df = pd.read_csv(BEST_PRED_PATH)
    feature_summary_df = pd.read_csv(FEATURE_SUMMARY_PATH)

    analysis_df = best_pred_df.merge(
        feature_summary_df[['name', 'frames_total', 'frames_valid', 'mean_conf', 'status']],
        on='name',
        how='left',
    )
    analysis_df['valid_ratio'] = analysis_df['frames_valid'] / analysis_df['frames_total'].clip(lower=1)
    display(analysis_df.head())
else:
    print('Required files for failure analysis are missing.')

,name,split,true_count,raw_pred_count,eval_pred_count,abs_error,frames_total,frames_valid,mean_conf,status,valid_ratio
0,stu6_60.mp4,train,35.0,37.453880,37.453880,2.453880,1740,1740,0.994441,ok,1.000000
1,stu2_69.mp4,train,21.0,21.897184,21.897184,0.897184,1950,1950,0.989398,ok,1.000000
2,stu1_72.mp4,train,21.0,23.997595,23.997595,2.997595,1439,1412,0.915094,ok,0.981237
3,stu3_65.mp4,train,6.0,7.152826,7.152826,1.152826,1109,1109,0.995579,ok,1.000000
4,stu8_67.mp4,train,3.0,2.350230,2.350230,0.649770,330,330,0.972717,ok,1.000000


In [22]:
if 'analysis_df' in globals():
    valid_analysis_df = analysis_df.loc[analysis_df['split'] == 'valid'].copy()
    display(valid_analysis_df.sort_values('abs_error', ascending=False).head(10))
else:
    print('analysis_df not available')

,name,split,true_count,raw_pred_count,eval_pred_count,abs_error,frames_total,frames_valid,mean_conf,status,valid_ratio
105,stu7_66.mp4,valid,13.0,5.363499,5.363499,7.636501,990,990,0.992587,ok,1.000000
106,stu6_61.mp4,valid,19.0,23.500055,23.500055,4.500055,1680,1680,0.995990,ok,1.000000
117,stu10_69.mp4,valid,39.0,43.368965,43.368965,4.368965,1920,1917,0.982843,ok,0.998437
115,test2349.mp4,valid,6.0,3.004511,3.004511,2.995489,128,128,0.963210,ok,1.000000
116,stu9_67.mp4,valid,0.0,2.891377,2.891377,2.891377,275,275,0.993504,ok,1.000000
109,stu10_63.mp4,valid,16.0,13.461786,13.461786,2.538214,1052,1040,0.864340,ok,0.988593
104,stu6_57.mp4,valid,15.0,17.323008,17.323008,2.323008,1559,1528,0.961495,ok,0.980115
113,stu7_65.mp4,valid,9.0,7.633377,7.633377,1.366623,711,597,0.769336,ok,0.839662
111,stu5_71.mp4,valid,19.0,17.668873,17.668873,1.331127,1031,1030,0.987552,ok,0.999030
108,train3921.mp4,valid,4.0,2.681303,2.681303,1.318697,300,300,0.955284,ok,1.000000


In [23]:
if 'valid_analysis_df' in globals() and not valid_analysis_df.empty:
    summary_cols = ['abs_error', 'frames_valid', 'mean_conf', 'valid_ratio']
    display(valid_analysis_df[summary_cols].corr())
else:
    print('valid_analysis_df not available or empty')

,abs_error,frames_valid,mean_conf,valid_ratio
abs_error,1.000000,0.208162,0.252758,0.248750
frames_valid,0.208162,1.000000,0.382647,0.363753
mean_conf,0.252758,0.382647,1.000000,0.986435
valid_ratio,0.248750,0.363753,0.986435,1.000000


In [24]:
if 'valid_analysis_df' in globals() and not valid_analysis_df.empty:
    display(
        valid_analysis_df.sort_values('abs_error', ascending=False)[
            ['name', 'true_count', 'raw_pred_count', 'eval_pred_count', 'abs_error', 'frames_valid', 'mean_conf', 'valid_ratio']
        ].head(10)
    )
else:
    print('valid_analysis_df not available or empty')

,name,true_count,raw_pred_count,eval_pred_count,abs_error,frames_valid,mean_conf,valid_ratio
105,stu7_66.mp4,13.0,5.363499,5.363499,7.636501,990,0.992587,1.000000
106,stu6_61.mp4,19.0,23.500055,23.500055,4.500055,1680,0.995990,1.000000
117,stu10_69.mp4,39.0,43.368965,43.368965,4.368965,1917,0.982843,0.998437
115,test2349.mp4,6.0,3.004511,3.004511,2.995489,128,0.963210,1.000000
116,stu9_67.mp4,0.0,2.891377,2.891377,2.891377,275,0.993504,1.000000
109,stu10_63.mp4,16.0,13.461786,13.461786,2.538214,1040,0.864340,0.988593
104,stu6_57.mp4,15.0,17.323008,17.323008,2.323008,1528,0.961495,0.980115
113,stu7_65.mp4,9.0,7.633377,7.633377,1.366623,597,0.769336,0.839662
111,stu5_71.mp4,19.0,17.668873,17.668873,1.331127,1030,0.987552,0.999030
108,train3921.mp4,4.0,2.681303,2.681303,1.318697,300,0.955284,1.000000


### How to read the failure analysis

- If the worst `valid` errors also have low `frames_valid`, low `mean_conf`, or low `valid_ratio`, then pose or feature quality is likely contributing to the counting error.
- If the worst errors happen even when confidence and valid support are strong, then the remaining issue is more likely in the model or target representation rather than upstream feature quality.
- This section should be used before deciding whether more squat tuning is justified or whether the project should move on to the multi-exercise branch.

Interpretation of the current correlation output:
- `abs_error` shows only weak positive correlation with `frames_valid` (`0.208`), `mean_conf` (`0.253`), and `valid_ratio` (`0.249`).
- This suggests the remaining validation mistakes are not strongly explained by weak pose support or low confidence.
- The strongest correlation in the table is between `mean_conf` and `valid_ratio` (`0.986`), which means those two quality indicators are nearly redundant in this summary.
- The practical conclusion is that the current best TCN's remaining errors are more likely tied to model behavior, motion ambiguity, or target variability than to poor pose extraction alone.